# Hotel Booking Cancellation Analysis

## Project Overview

This project analyzes hotel booking data to understand the characteristics associated with booking cancellations.

Hotel cancellations can affect occupancy planning, revenue management, and operational decision-making. The purpose of this analysis is to identify meaningful patterns in cancellation behavior and translate those findings into business insights and recommendations.

## Dataset

The dataset used in this project is the **Hotel Booking Demand** dataset, containing 119,390 hotel booking records across 32 variables. Each row represents an individual hotel booking and includes information about the hotel, booking characteristics, customer history, deposit type, market segment, scheduled arrival, and final reservation outcome.

**Source:** [Hotel Booking Demand Dataset on Kaggle](https://www.kaggle.com/datasets/jessemostipak/hotel-booking-demand)

The primary outcome examined in this project is `is_canceled`, which indicates whether a booking was canceled or resulted in a no-show.

## Business Question

**What booking characteristics are associated with hotel cancellations, and what can hotels learn from these patterns to support cancellation-risk management?**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('../data/raw/hotel_bookings.csv')

In [ ]:
df.head(10)

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isna().sum()

# Analysis

The analysis focuses on understanding hotel booking cancellations and identifying the booking characteristics most strongly associated with cancellation.

The investigation examines cancellation rates across hotel types, lead time, deposit type, previous cancellation history, market segments, cancellation timing, and seasonality. The goal is to identify meaningful patterns that can provide useful insights into cancellation behavior.

## 1. Initial Data Overview

The dataset contains 119,390 hotel bookings across 32 variables. Most variables are complete, while missing values are concentrated mainly in `agent`, `company`, and `country`. The `children` column contains only four missing values.

In [ ]:
df['is_canceled'].value_counts()

In [ ]:
df['is_canceled'].value_counts(normalize=True) * 100

In [ ]:
cancellation_rate = (
    df['is_canceled']
    .value_counts(normalize=True)
    .mul(100)
    .rename(index={0: 'Not Canceled', 1: 'Canceled'})
)

plt.figure(figsize=(4, 3))

bars = plt.bar(
    cancellation_rate.index,
    cancellation_rate.values,
    width=0.6
)

plt.bar_label(bars, fmt='%.1f%%', padding=3)

plt.xlabel('Reservation Outcome')
plt.ylabel('Percentage of Bookings')
plt.title('Overall Hotel Booking Cancellation Rate')

plt.ylim(0, 100)

plt.savefig(
    '../outputs/charts/cancellation_rate_overall.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

## 2. Cancellation Overview

The dataset contains 119,390 hotel bookings. Overall, 37.04% of bookings were canceled, while 62.96% were not canceled. This overall cancellation rate serves as the baseline for comparing different booking groups throughout the analysis.

In [ ]:
df.groupby('hotel')['is_canceled'].mean() * 100

In [ ]:
df['hotel'].value_counts()

In [ ]:
hotel_cancellation = (
    df.groupby('hotel')['is_canceled']
    .mean()
    .mul(100)
)

plt.figure(figsize=(4, 3))

bars = plt.bar(
    hotel_cancellation.index,
    hotel_cancellation.values,
    width=0.6
)

plt.bar_label(bars, fmt='%.1f%%', padding=3)

plt.xlabel('Hotel Type')
plt.ylabel('Cancellation Rate (%)')
plt.title('Cancellation Rate by Hotel Type')

plt.ylim(0, 50)

plt.savefig('../outputs/charts/cancellation_rate_by_hotel.png', dpi=300, bbox_inches='tight')

plt.show()

## 3. Cancellation by Hotel Type

City Hotel recorded a higher cancellation rate than Resort Hotel, at 41.73% compared with 27.76%.

In [ ]:
df.groupby('is_canceled')['lead_time'].agg(['count', 'mean', 'median', 'min', 'max'])

In [ ]:
bins = [0, 30, 60, 90, 180, 365, float('inf')]
labels = ['0–30', '31–60', '61–90', '91–180', '181–365', '366+']

df['lead_time_group'] = pd.cut(
    df['lead_time'],
    bins=bins,
    labels=labels,
    include_lowest=True
)

In [ ]:
df.groupby('lead_time_group', observed=True)['is_canceled'].agg(
    ['count', 'mean']
)

In [ ]:
df.groupby('lead_time_group', observed=True)['is_canceled'].mean() * 100

## 4. Lead Time and Cancellation

Cancellation rates increased substantially as the time between booking and scheduled arrival increased. Bookings made 0–30 days before arrival had an 18.56% cancellation rate, compared with 67.66% for bookings made 366 or more days in advance. 

One possible explanation is that bookings made further in advance give customers more time to change their plans or cancel before arrival. However, this analysis cannot determine whether this explains the observed relationship.

In [ ]:
lead_time_cancel = (
    df.groupby('lead_time_group', observed=True)['is_canceled']
      .mean()
      .mul(100)
)

lead_time_cancel

In [ ]:
plt.figure(figsize=(7, 4))

bars = plt.bar(
    lead_time_cancel.index,
    lead_time_cancel.values,
    width=0.7
)

plt.bar_label(bars, fmt='%.1f%%', padding=3)

plt.xlabel('Lead Time (days)')
plt.ylabel('Cancellation Rate (%)')
plt.title('Cancellation Rate by Lead Time')

plt.ylim(0, 80)

plt.savefig('../outputs/charts/cancellation_rate_by_lead_time.png', dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
df.groupby(['hotel', 'lead_time_group'], observed=True)['is_canceled'].mean() * 100

The increase in cancellation rates with longer lead times was present in both hotel types, although City Hotel consistently recorded higher cancellation rates across the lead-time groups.

In [ ]:
df['deposit_type'].value_counts()

In [ ]:
df.groupby('deposit_type')['is_canceled'].mean() * 100

In [ ]:
deposit_cancellation = (
    df.groupby('deposit_type')['is_canceled']
    .mean()
    .mul(100)
)

plt.figure(figsize=(6, 4))

bars = plt.bar(
    deposit_cancellation.index,
    deposit_cancellation.values,
    width=0.7
)

plt.bar_label(bars, fmt='%.1f%%', padding=3)

plt.xlabel('Deposit Type')
plt.ylabel('Cancellation Rate (%)')
plt.title('Cancellation Rate by Deposit Type')

plt.ylim(0, 110)

plt.savefig('../outputs/charts/cancellation_rate_by_deposit_type.png', dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
pd.crosstab(df['deposit_type'], df['is_canceled'])

## 5. Deposit Type and Cancellation

Deposit type showed the strongest observed difference in cancellation behavior. Non Refund bookings had a 99.36% cancellation rate, compared with 28.38% for No Deposit bookings and 22.22% for Refundable bookings.

The crosstab confirms that 14,494 of the 14,587 Non Refund bookings were recorded as canceled according to `is_canceled`.

The large difference between deposit types may reflect factors beyond the deposit policy itself, such as booking channels, customer behavior, pricing, or other characteristics associated with these bookings. The available data does not allow these possibilities to be separated.

In [ ]:
df.groupby('deposit_type')['lead_time'].agg(['count', 'mean', 'median'])

In [ ]:
df.groupby('deposit_type')['market_segment'].value_counts()

In [ ]:
df.groupby('market_segment')['is_canceled'].agg(['count', 'mean'])

In [ ]:
df.groupby('market_segment')['is_canceled'].mean().mul(100).sort_values(ascending=False)

In [ ]:
market_segment_cancellation = (
    df.groupby('market_segment')['is_canceled']
    .mean()
    .mul(100)
    .sort_values()
)

plt.figure(figsize=(7, 5))

bars = plt.barh(
    market_segment_cancellation.index,
    market_segment_cancellation.values,
    height=0.7
)

plt.bar_label(bars, fmt='%.1f%%', padding=3)

plt.xlabel('Cancellation Rate (%)')
plt.ylabel('Market Segment')
plt.title('Cancellation Rate by Market Segment')

plt.xlim(0, 70)

plt.savefig(
    '../outputs/charts/cancellation_rate_by_market_segment.png', 
    dpi=300, 
    bbox_inches='tight'
    )

plt.show()

## 6. Market Segment

Cancellation behavior varied substantially across market segments. Groups bookings had the highest cancellation rate among the major segments at 61.06%, while Direct bookings had a much lower rate of 15.34%.

The Undefined category contains only two bookings and is therefore not meaningful for interpretation.

These differences may reflect differences in customer behavior or booking processes across market segments, rather than the market segment itself causing cancellations.

In [ ]:
df['has_previous_cancellation'] = (df['previous_cancellations'] > 0).astype(int)

In [ ]:
df.groupby('has_previous_cancellation')['is_canceled'].agg(['count', 'mean'])

In [ ]:
df.groupby('has_previous_cancellation')['is_canceled'].mean().mul(100)

In [ ]:
previous_cancellation_rate = (
    df.groupby('has_previous_cancellation')['is_canceled']
    .mean()
    .mul(100)
    .rename(index={
        0: 'No Previous Cancellation',
        1: 'Previous Cancellation'
    })
)

plt.figure(figsize=(4, 3))

bars = plt.bar(
    previous_cancellation_rate.index,
    previous_cancellation_rate.values,
    width=0.6
)

plt.bar_label(bars, fmt='%.1f%%', padding=3)

plt.xlabel('Previous Cancellation History')
plt.ylabel('Cancellation Rate (%)')
plt.title('Cancellation Rate by Previous Cancellation History')

plt.ylim(0, 100)

plt.savefig(
    '../outputs/charts/cancellation_rate_by_previous_cancellation_history.png', 
    dpi=300, 
    bbox_inches='tight'
)

plt.show()

## 7. Previous Cancellation History

Bookings from customers with at least one previous cancellation had a 91.64% cancellation rate, compared with 33.91% for bookings with no previous cancellation history.

In [ ]:
df.groupby(
    ['has_previous_cancellation', 'deposit_type'],
    observed=True
)['is_canceled'].mean().mul(100)

In [ ]:
df['high_risk_profile'] = (
    (df['has_previous_cancellation'] == 1) &
    (df['deposit_type'] == 'Non Refund') &
    (df['lead_time_group'].isin(['181–365', '366+']))
).astype(int)

In [ ]:
df.groupby('high_risk_profile')['is_canceled'].agg(['count', 'mean'])

## 8. High-Risk Booking Profile

A particularly high-risk segment was identified by combining three characteristics: previous cancellation history, a Non Refund deposit, and a lead time of at least 181 days.

Among the 2,405 bookings matching this profile, 100% were recorded as canceled or no-show. This provides supporting evidence that the combination of these characteristics is strongly associated with unfavorable reservation outcomes in this dataset.

This should be interpreted as an observed high-risk segment rather than evidence that these characteristics cause cancellation.

In [ ]:
df.groupby(
    ['has_previous_cancellation', 'deposit_type']
)['lead_time_group'].value_counts(normalize=True)

In [ ]:
df['arrival_date_month'].unique()

In [ ]:
df['arrival_date'] = pd.to_datetime(
    df['arrival_date_year'].astype(str) + '-' +
    df['arrival_date_month'] + '-' +
    df['arrival_date_day_of_month'].astype(str),
    format='%Y-%B-%d'
)

In [ ]:
df[['arrival_date_year', 'arrival_date_month',
    'arrival_date_day_of_month', 'arrival_date']].head()

In [ ]:
df['reservation_status_date'] = pd.to_datetime(
    df['reservation_status_date']
)

In [ ]:
df[['reservation_status_date', 'reservation_status']].head()

In [ ]:
canceled_df = df[df['reservation_status'] == 'Canceled'].copy()

In [ ]:
canceled_df.shape

In [ ]:
canceled_df['days_before_arrival'] = (
    canceled_df['arrival_date'] -
    canceled_df['reservation_status_date']
).dt.days

In [ ]:
canceled_df['days_before_arrival'].describe()

## 9. Cancellation Timing

Among the 43,017 bookings with a final reservation status of `Canceled`, the median cancellation occurred 56 days before the scheduled arrival date. The mean was higher at 88 days, indicating that some cancellations occurred substantially further in advance.

In [ ]:
cancellation_timing_bins = [0, 7, 30, 60, 90, 180, float('inf')]
cancellation_timing_labels = [
    '0–7 days',
    '8–30 days',
    '31–60 days',
    '61–90 days',
    '91–180 days',
    '181+ days'
]

canceled_df['cancellation_timing_group'] = pd.cut(
    canceled_df['days_before_arrival'],
    bins=cancellation_timing_bins,
    labels=cancellation_timing_labels,
    include_lowest=True
)

In [ ]:
canceled_df['cancellation_timing_group'].value_counts().sort_index()

In [ ]:
canceled_df['cancellation_timing_group'].value_counts(
    normalize=True
).sort_index() * 100

In [ ]:
cancellation_timing = (
    canceled_df['cancellation_timing_group']
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
)

plt.figure(figsize=(7, 4))

bars = plt.bar(
    cancellation_timing.index,
    cancellation_timing.values,
    width=0.7
)

plt.bar_label(bars, fmt='%.1f%%', padding=3)

plt.xlabel('Days Before Arrival')
plt.ylabel('Percentage of Cancellations (%)')
plt.title('Distribution of Cancellation Timing')

plt.ylim(0, 30)

plt.savefig(
    '../outputs/charts/distribution_of_cancellation_timing.png', 
    dpi=300, 
    bbox_inches='tight'
)

plt.show()

## 10. Distribution of Cancellation Timing

Cancellations were distributed throughout the booking lifecycle rather than being concentrated only immediately before arrival. About 33.71% of cancellations occurred within 30 days of the scheduled arrival date, while approximately 36.14% occurred 91 or more days before arrival.

This suggests that cancellation management may need to begin well before the scheduled arrival date rather than focusing only on last-minute cancellations.

In [ ]:
canceled_df.groupby(
    'deposit_type'
)['days_before_arrival'].agg(['count', 'mean', 'median'])

## 11. Cancellation Timing by Deposit Type

Non Refund cancellations occurred substantially earlier than No Deposit cancellations. The median cancellation occurred 102 days before arrival for Non Refund bookings, compared with 37 days for No Deposit bookings.

The Refundable category contains only 35 canceled bookings and is therefore too small for a meaningful comparison.

In [ ]:
df.groupby('arrival_date_month')['is_canceled'].agg(['count', 'mean'])

In [ ]:
month_order = [
    'January', 'February', 'March', 'April', 'May', 'June',
    'July', 'August', 'September', 'October', 'November', 'December'
]

df['arrival_date_month'] = pd.Categorical(
    df['arrival_date_month'],
    categories=month_order,
    ordered=True
)

In [ ]:
monthly_cancellation = (
    df.groupby('arrival_date_month', observed=True)['is_canceled']
      .agg(['count', 'mean'])
)

monthly_cancellation['mean'] = monthly_cancellation['mean'] * 100

monthly_cancellation

In [ ]:
plt.figure(figsize=(8, 4))

plt.plot(
    monthly_cancellation.index,
    monthly_cancellation['mean'],
    marker='o'
)

plt.xlabel('Arrival Month')
plt.ylabel('Cancellation Rate (%)')
plt.title('Cancellation Rate by Arrival Month')

plt.ylim(0, 50)

plt.xticks(rotation=45)

plt.savefig(
    '../outputs/charts/cancellation_rate_by_arrival_month.png', 
    dpi=300, 
    bbox_inches='tight'
)

plt.show()

## 12. Seasonality

Cancellation rates varied by the month of scheduled arrival, ranging from 30.48% in January to 41.46% in June. This variation is noticeable but smaller than the differences observed for deposit type, previous cancellation history, and lead time.

In [ ]:
df.groupby(
    ['hotel', 'arrival_date_month'],
    observed=True
)['is_canceled'].mean().mul(100)

The seasonal pattern differed between the two hotel types. City Hotel maintained relatively high cancellation rates throughout the year, while Resort Hotel showed a clearer seasonal pattern, increasing from 14.82% in January to 33.45% in August.

In [ ]:
risk_by_deposit_history = (
    df.groupby(
        ['has_previous_cancellation', 'deposit_type'],
        observed=True
    )['is_canceled']
    .agg(['count', 'mean'])
)

risk_by_deposit_history['mean'] *= 100

risk_by_deposit_history

In [ ]:
combined_risk = (
    risk_by_deposit_history
    .reset_index()
)

combined_risk['has_previous_cancellation'] = combined_risk[
    'has_previous_cancellation'
].map({
    0: 'No Previous Cancellation',
    1: 'Previous Cancellation'
})

pivot_risk = combined_risk.pivot(
    index='deposit_type',
    columns='has_previous_cancellation',
    values='mean'
)

plt.figure(figsize=(8, 5))

bars = pivot_risk.plot(
    kind='bar',
    figsize=(8, 5),
    width=0.6
)

plt.ylabel('Cancellation Rate (%)')
plt.xlabel('Deposit Type')
plt.title('Cancellation Rate by Deposit Type and Previous Cancellation History')

plt.ylim(0, 110)

plt.legend(title='Previous Cancellation History')

plt.savefig(
    '../outputs/charts/cancellation_rate_by_deposit_type_and_previous_cancellation_history.png', 
    dpi=300, 
    bbox_inches='tight'
)

plt.show()

## 13. Combined Risk Factors

The strongest combined pattern emerged when previous cancellation history was considered alongside deposit type.

Among bookings with no previous cancellation history, No Deposit bookings had a 26.91% cancellation rate, compared with 99.15% for Non Refund bookings.

Among bookings with previous cancellation history, the cancellation rate was 80.90% for No Deposit bookings and 100% for Non Refund bookings.

This indicates that the strong association between cancellation and deposit type is also observed when previous cancellation history is considered, while previous cancellation history remains strongly associated with cancellation across the deposit categories examined.

## 14. Key Findings

The analysis identified three particularly strong factors associated with booking cancellations:

- **Deposit type:** Non Refund bookings had a 99.36% cancellation rate.
- **Previous cancellation history:** bookings with at least one previous cancellation had a 91.64% cancellation rate.
- **Lead time:** cancellation rates increased from 18.56% for bookings made 0–30 days before arrival to 67.66% for bookings made 366 or more days in advance.

Additional differences were observed across market segments, cancellation timing, hotel type, and arrival seasonality. Together, these findings show that cancellation behavior varies substantially according to the characteristics and history of a booking.

# Business Insights

The analysis shows that cancellation risk is not evenly distributed across hotel bookings. Several booking characteristics were associated with substantially different cancellation rates, with deposit type, previous cancellation history, and lead time showing the strongest patterns.

## 1. Deposit Type Is Strongly Associated With Cancellation

Non Refund bookings had an exceptionally high observed cancellation rate of 99.36%, compared with 28.38% for No Deposit bookings and 22.22% for Refundable bookings.

This makes deposit type one of the strongest indicators of cancellation behavior in the dataset. However, the result should be interpreted as an association rather than evidence that the deposit policy itself causes cancellations.

## 2. Previous Cancellation History Is a Strong Risk Indicator

Bookings from customers with at least one previous cancellation had a 91.64% cancellation rate, compared with 33.91% among bookings with no previous cancellation history.

This suggests that previous booking behavior can provide useful information when assessing the risk of a future booking cancellation.

## 3. Longer Lead Times Are Associated With Higher Cancellation Risk

Cancellation rates increased consistently as the time between booking and scheduled arrival became longer. The rate increased from 18.56% for bookings made 0–30 days before arrival to 67.66% for bookings made 366 or more days in advance.

This suggests that cancellation risk can be identified relatively early in the booking lifecycle rather than only when the arrival date is approaching.

## 4. Multiple Risk Characteristics Can Overlap

The combination of previous cancellation history, Non Refund deposit type, and a lead time of at least 181 days identified a particularly high-risk group.

Among the 2,405 bookings matching these characteristics, 100% were recorded as canceled or no-show.

This provides additional supporting evidence that the strongest individual risk factors identified in the analysis can overlap within the same bookings. The profile should be viewed as an observed high-risk segment in this dataset, not as evidence that these characteristics cause unfavorable outcomes.

## 5. Cancellations Occur Throughout the Booking Lifecycle

Among bookings with a final reservation status of `Canceled`, the median cancellation occurred 56 days before the scheduled arrival date.

Only about 33.71% of cancellations occurred within 30 days of arrival, while approximately 36.14% occurred 91 or more days before arrival.

This suggests that cancellation-management efforts should not focus exclusively on the final days before arrival. Hotels may benefit from monitoring cancellation risk earlier in the reservation lifecycle.

# Business Recommendations

The findings suggest several ways hotels could use booking characteristics to improve cancellation-risk management. These recommendations are based on observed associations in the dataset and should be validated with operational data before being implemented.

## 1. Identify High-Risk Bookings Earlier

Hotels could use information available at the time of booking, such as lead time, deposit type, and previous cancellation history, to identify bookings that may have a higher likelihood of cancellation.

Early identification would allow staff to monitor potentially high-risk bookings throughout the reservation lifecycle rather than waiting until the arrival date approaches.

## 2. Prioritize Bookings With Multiple Risk Characteristics

Bookings that combine several high-risk characteristics could receive greater attention than bookings with only one risk indicator.

In this analysis, the combination of previous cancellation history, Non Refund deposit type, and a lead time of at least 181 days identified a particularly high-risk segment. Hotels could use similar combinations of booking characteristics to prioritize monitoring and intervention.

## 3. Monitor Cancellation Risk Throughout the Booking Lifecycle

Because cancellations occurred throughout the booking lifecycle, cancellation-management efforts should not be limited to the final days before arrival.

Hotels could establish periodic monitoring of higher-risk bookings, allowing them to identify changes in booking behavior and respond earlier when appropriate.

## 4. Review Cancellation and Deposit Policies

The exceptionally high cancellation rate observed among Non Refund bookings warrants further investigation.

Hotels could review this booking category to understand whether factors such as booking channels, customer behavior, pricing, policy structure, or data-recording practices contribute to the observed pattern.

The goal should not be to assume that the Non Refund policy causes cancellations, but to investigate why this category has such a different cancellation outcome from other deposit types.

## 5. Use Booking Characteristics to Support Operational Decision-Making

Cancellation risk indicators could potentially be incorporated into reservation-management processes to help hotels prioritize attention, manage expected occupancy, and make more informed decisions about inventory and booking policies.

Any such approach should be tested against future booking data before being relied upon for operational decisions.

# Limitations

## 1. The Analysis Shows Associations, Not Causation

This analysis identifies relationships between booking characteristics and cancellation outcomes, but it does not establish that one factor causes another.

For example, Non Refund bookings had an exceptionally high observed cancellation rate, but the analysis cannot determine whether the deposit policy itself caused this outcome. Other factors, such as lead time, market segment, distribution channel, or booking practices, may also contribute to the observed relationship.

## 2. Results Are Specific to This Dataset

The findings are based on the hotel booking data used in this analysis and should not automatically be generalized to other hotels, locations, or time periods.

For example, the 99.36% cancellation rate observed for Non Refund bookings is a characteristic of this dataset and may not represent cancellation behavior in other hotel environments.

## 3. Some Categories Have Small Sample Sizes

Certain categories contain relatively few observations, making their results less reliable for comparison.

For example, the Refundable deposit category contains only 162 bookings, and the Undefined market segment contains only two bookings. Findings involving these small groups should therefore be interpreted with caution.

## 4. Limited Information About Booking Creation and Operational Processes

The dataset does not include a reservation creation date or detailed information about hotel operational decisions, booking management, or customer motivations.

As a result, the analysis cannot determine why certain booking characteristics are associated with higher cancellation rates or explain the operational processes behind the observed patterns.

## 5. The High-Risk Profile Is an Observed Pattern

The combined high-risk profile was created from characteristics identified during the analysis and showed a 100% unfavorable outcome within the 2,405 matching bookings.

However, this does not mean that every future booking with these characteristics will necessarily be canceled or become a no-show. The result should be interpreted as an observed pattern within this dataset rather than a validated predictive rule.

# Final Conclusion

This analysis examined the factors associated with hotel booking cancellations using 119,390 booking records.

The results showed that cancellation behavior was not evenly distributed across bookings. The strongest observed differences were associated with deposit type, previous cancellation history, and lead time. Non Refund bookings and bookings from customers with previous cancellations had particularly high cancellation rates, while cancellation rates also increased substantially as lead time increased.

The analysis also showed that multiple high-risk characteristics can overlap within the same bookings, identifying a segment with particularly unfavorable reservation outcomes. In addition, cancellations occurred throughout the booking lifecycle rather than being concentrated only immediately before arrival.

Overall, the findings suggest that hotels may benefit from treating cancellation risk as something that can vary across bookings rather than applying the same approach to every reservation. Booking characteristics and customer history could potentially help identify higher-risk bookings earlier and support more targeted reservation-management strategies.

However, these findings represent associations observed within this dataset and should not be interpreted as evidence of causal relationships or as predictions that will necessarily apply to other hotels or future bookings.